# Dictionary overlay

Score a document collection against a user-defined dictionary (lexicon), map the
dictionary's groups onto a science backbone (taxonomy), and see where the
collection's attention falls.

This notebook uses the built-in synthetic sample corpus, SUBSIDE lexicon, and
SUBSIDE backbone, so it runs offline. Swap in your own files in the first cell.

In [ ]:
from semantic_sensemaking import load_corpus, load_lexicon, load_taxonomy, sample_corpus_path
from semantic_sensemaking.core.config import ScoringConfig
from semantic_sensemaking.workflows.dictionary import analyze

corpus = load_corpus(sample_corpus_path())   # or load_corpus("path/to/docs")
lexicon = load_lexicon()                     # or load_lexicon("my_lexicon.csv")
taxonomy = load_taxonomy()                   # or load_taxonomy("science_backbone.json")
corpus.describe()

## Inspect the inputs

A lexicon is a set of named groups, each with the terms that signal it. The
taxonomy is the backbone the groups are projected onto.

In [ ]:
lexicon.to_mapping()

In [ ]:
taxonomy.domains()

## Analyze

`longest_match` counts overlapping terms such as *land subsidence* and
*subsidence* once; `subdiscipline_aware` spreads each group across the
subdisciplines whose names it mentions. The defaults (`auto`, `legacy`)
reproduce the original SUBSIDE notebook exactly.

In [ ]:
scoring = ScoringConfig(match_mode="longest_match", mapping_method="subdiscipline_aware")
result = analyze(corpus, lexicon, taxonomy, scoring)
result.scores.group_totals().head(10)

In [ ]:
result.mapping.to_frame()

In [ ]:
result.projection.domain_totals()

## Base vs. overlay

`base_pct` is the share of lexicon groups whose primary domain is each domain;
`overlay_pct` is the share of the corpus's matches projected there. Large
deltas show where the collection's attention differs from the dictionary's
design.

In [ ]:
result.comparison

## Figures

Requires the `viz` extra (`pip install "semantic-sensemaking[viz]"`).

In [ ]:
from semantic_sensemaking.viz import render_backbone_overlay, render_distribution_comparison

render_backbone_overlay(result.to_overlay_data())

In [ ]:
render_distribution_comparison(result.comparison)

## One document

In [ ]:
doc_scores, doc_projection = result.document_projection(0)
doc_projection.domain_totals()

## Next steps

- Run the same analysis from the command line, writing tables, figures, a
  summary, and a manifest: `semantic-sensemaking dictionary --config configs/dictionary-example.yaml`
- Draft a lexicon for a new problem with the `formulation` workflow, or find
  frames to count with the `narrative` workflow.